# 🌱 CSIRO Image2Biomass - Complete Solution

**Multi-Modal Deep Learning Pipeline**

- CNN Backbones: EfficientNet, ConvNeXt, Swin Transformer
- NDVI & Metadata Integration
- Test Time Augmentation
- 5-Fold Cross-Validation
- Model Ensemble

---

## ⚙️ Notebook Settings

**IMPORTANT**: Configure these settings:
- Accelerator: **GPU T4 x2** or **GPU P100**
- Internet: **ON** (to clone from GitHub)
- Persistence: **Files only**

## 📥 Step 1: Setup and Installation

In [ ]:
%%time
# Clone repository from GitHub
!git clone https://github.com/EmreUludasdemir/CSIRO---Image2Biomass-.git
%cd CSIRO---Image2Biomass-

# Install required packages
!pip install -q timm==0.9.12 albumentations==1.3.1 pytorch-lightning==2.0.0

print("\n✅ Setup complete!")

In [ ]:
# Check GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## ⚙️ Step 2: Configure for Kaggle Environment

In [ ]:
import yaml
import os
import sys

# Add src to path
sys.path.append('/kaggle/working/CSIRO---Image2Biomass-/src')

# Load original config
with open('configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Update paths for Kaggle
config['data']['data_dir'] = '/kaggle/input/csiro-biomass'
config['data']['image_dir'] = 'images'  # Adjust based on actual data structure
config['paths']['checkpoint_dir'] = '/kaggle/working/checkpoints'
config['paths']['best_model_dir'] = '/kaggle/working/best_models'
config['paths']['submission_dir'] = '/kaggle/working/submissions'

# Training optimization for Kaggle
config['training']['batch_size'] = 16  # Adjust based on GPU memory
config['training']['num_epochs'] = 20  # Adjust based on time limit
config['training']['mixed_precision'] = True
config['logging']['use_wandb'] = False  # Disable wandb in Kaggle

# Save Kaggle config
os.makedirs('configs', exist_ok=True)
with open('configs/config_kaggle.yaml', 'w') as f:
    yaml.dump(config, f)

print("✅ Kaggle configuration ready!")
print(f"\nData directory: {config['data']['data_dir']}")
print(f"Checkpoint directory: {config['paths']['checkpoint_dir']}")

In [ ]:
# Check data availability
import pandas as pd

print("Checking data files...\n")

# List available data
print("Available datasets:")
!ls /kaggle/input/

print("\nData structure:")
!ls -lh /kaggle/input/csiro-biomass/

# Load and check train data
train_path = '/kaggle/input/csiro-biomass/train.csv'
if os.path.exists(train_path):
    train_df = pd.read_csv(train_path)
    print(f"\n✅ Train data loaded: {len(train_df)} samples")
    print(f"Columns: {list(train_df.columns)}")
    print("\nFirst few rows:")
    print(train_df.head())
else:
    print("\n⚠️ Train data not found! Please add the competition data.")

## 📊 Step 3: Exploratory Data Analysis (Optional)

In [ ]:
%%time
# Run EDA
!python src/eda.py \
    --data-dir /kaggle/input/csiro-biomass \
    --train-csv train.csv \
    --image-dir images \
    --save-dir /kaggle/working/eda_plots

In [ ]:
# Display EDA plots
from IPython.display import Image, display
import os

plot_dir = '/kaggle/working/eda_plots'
if os.path.exists(plot_dir):
    for plot_file in os.listdir(plot_dir):
        if plot_file.endswith('.png'):
            print(f"\n{plot_file}:")
            display(Image(os.path.join(plot_dir, plot_file)))

## 🏋️ Step 4: Model Training

Choose one of the following training options:

### Option A: Quick Training (Single Fold for Testing)

In [ ]:
%%time
# Quick training for testing (single fold, fewer epochs)
from train import train_fold
from sklearn.model_selection import train_test_split
import pandas as pd

# Load config
with open('configs/config_kaggle.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Quick test settings
config['training']['num_epochs'] = 5
config['data']['n_folds'] = 1

# Load data
train_df = pd.read_csv('/kaggle/input/csiro-biomass/train.csv')
train_data, valid_data = train_test_split(train_df, test_size=0.2, random_state=42)

# Train
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
best_score, model = train_fold(
    fold=0,
    train_df=train_data,
    valid_df=valid_data,
    config=config,
    device=device
)

print(f"\n✅ Quick training complete! Best RMSE: {best_score:.4f}")

### Option B: Full Training (5-Fold Cross-Validation)

In [ ]:
%%time
# Full training with cross-validation
!python src/train.py --config configs/config_kaggle.yaml

## 🔍 Step 5: Check Training Results

In [ ]:
# List trained models
checkpoint_dir = '/kaggle/working/checkpoints'
if os.path.exists(checkpoint_dir):
    print("Trained models:")
    !ls -lh {checkpoint_dir}
    
    # Load and show best scores
    import torch
    for checkpoint_file in sorted(os.listdir(checkpoint_dir)):
        if checkpoint_file.startswith('best_model'):
            checkpoint_path = os.path.join(checkpoint_dir, checkpoint_file)
            checkpoint = torch.load(checkpoint_path, map_location='cpu')
            print(f"\n{checkpoint_file}:")
            print(f"  Epoch: {checkpoint['epoch']}")
            print(f"  Best RMSE: {checkpoint['score']:.4f}")
else:
    print("⚠️ No checkpoints found!")

## 🔮 Step 6: Generate Predictions

In [ ]:
%%time
# Generate submission
from inference import create_submission

create_submission(
    config_path='configs/config_kaggle.yaml',
    test_csv_path='/kaggle/input/csiro-biomass/test.csv',
    output_path='/kaggle/working/submission.csv'
)

print("\n✅ Submission generated!")

## 📊 Step 7: Validate Submission

In [ ]:
import pandas as pd
import numpy as np

# Load submission
submission = pd.read_csv('/kaggle/working/submission.csv')

print("Submission validation:\n")
print(f"Shape: {submission.shape}")
print(f"Columns: {list(submission.columns)}")
print(f"\nFirst few predictions:")
print(submission.head(10))

print(f"\nPrediction statistics:")
print(submission['biomass'].describe())

print(f"\nNull values: {submission.isnull().sum().sum()}")
print(f"Infinite values: {np.isinf(submission['biomass']).sum()}")

# Check for negative predictions
if (submission['biomass'] < 0).any():
    print(f"\n⚠️ Warning: {(submission['biomass'] < 0).sum()} negative predictions found!")
else:
    print("\n✅ All predictions are positive")

print("\n✅ Submission is ready for upload!")

## 📈 Step 8: Visualize Predictions (Optional)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram
axes[0].hist(submission['biomass'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Predicted Biomass')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Predictions')
axes[0].axvline(submission['biomass'].mean(), color='r', linestyle='--', 
                label=f'Mean: {submission["biomass"].mean():.2f}')
axes[0].legend()

# Box plot
axes[1].boxplot(submission['biomass'])
axes[1].set_ylabel('Predicted Biomass')
axes[1].set_title('Box Plot of Predictions')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/prediction_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("Plot saved to /kaggle/working/prediction_distribution.png")

## 💾 Step 9: Save Important Files

In [ ]:
# Create a summary of the run
import json
from datetime import datetime

summary = {
    'timestamp': datetime.now().isoformat(),
    'config': config,
    'submission_stats': {
        'num_samples': len(submission),
        'mean_prediction': float(submission['biomass'].mean()),
        'std_prediction': float(submission['biomass'].std()),
        'min_prediction': float(submission['biomass'].min()),
        'max_prediction': float(submission['biomass'].max())
    }
}

with open('/kaggle/working/run_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("✅ Run summary saved to /kaggle/working/run_summary.json")

# List all output files
print("\nOutput files:")
!ls -lh /kaggle/working/

## 🎯 Next Steps

1. **Download submission**: Get `submission.csv` from the Output tab
2. **Submit to competition**: Go to competition page and submit
3. **Iterate**: Based on leaderboard score, adjust hyperparameters
4. **Ensemble**: Try different models and ensemble them

---

## 🔧 Troubleshooting

### Out of Memory
```python
# Reduce batch size
config['training']['batch_size'] = 8
config['training']['accumulation_steps'] = 4
```

### Timeout
```python
# Reduce epochs or folds
config['training']['num_epochs'] = 10
config['data']['n_folds'] = 3
```

---

**Good luck!** 🚀